## Data Cleaning

In [1]:
# Ensure working directory is set to project root
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Imports for numerical operations, data processing, and path verification
import os
import numpy as np
import pandas as pd

# Load freshly saved metadata CSV to ensure clean in-memory state
csv_filepath = "data/processed/utkface_metadata.csv"
df = pd.read_csv(csv_filepath)

# Output baseline row and column count before applying cleaning checks
print(f"Loaded metadata CSV from '{csv_filepath}'")
print(f"Before cleaning baseline shape: {df.shape}")

Loaded metadata CSV from 'data/processed/utkface_metadata.csv'
Before cleaning baseline shape: (23705, 5)


In [2]:
# Check 1: Check for exact full-row duplicate records
full_duplicates_count = df.duplicated().sum()
print(f"Full-row exact duplicates count: {full_duplicates_count}")

# Check 2: Check for duplicate image_name entries
name_duplicates_count = df.duplicated(subset=['image_name']).sum()
print(f"Duplicate image_name entries count: {name_duplicates_count}")

Full-row exact duplicates count: 0
Duplicate image_name entries count: 0


In [3]:
# Check 1: Count and display rows with impossible age values (< 0 or > 120)
invalid_age_df = df[(df['age'] < 0) | (df['age'] > 120)]
print(f"Invalid age values count (<0 or >120): {len(invalid_age_df)}")
if len(invalid_age_df) > 0:
    print("Invalid age records found:")
    print(invalid_age_df)

# Check 2: Count and display rows with gender values outside [0, 1]
invalid_gender_df = df[~df['gender'].isin([0, 1])]
print(f"Invalid gender values count (not in [0, 1]): {len(invalid_gender_df)}")
if len(invalid_gender_df) > 0:
    print("Invalid gender records found:")
    print(invalid_gender_df)

# Check 3: Count and display rows with race values outside [0, 1, 2, 3, 4]
invalid_race_df = df[~df['race'].isin([0, 1, 2, 3, 4])]
print(f"Invalid race values count (not in [0, 1, 2, 3, 4]): {len(invalid_race_df)}")
if len(invalid_race_df) > 0:
    print("Invalid race records found:")
    print(invalid_race_df)

Invalid age values count (<0 or >120): 0
Invalid gender values count (not in [0, 1]): 0
Invalid race values count (not in [0, 1, 2, 3, 4]): 0


In [4]:
# Verify that every image filepath in the DataFrame actually exists on disk
missing_filepaths_df = df[~df['filepath'].apply(os.path.exists)]
print(f"Missing / broken image filepaths count on disk: {len(missing_filepaths_df)}")
if len(missing_filepaths_df) > 0:
    print("Missing file records:")
    print(missing_filepaths_df)

Missing / broken image filepaths count on disk: 0


### Data Cleaning Decision Logic Summary

Based on the empirical verification checks performed above:
1. **Duplicates:** 0 full-row duplicates and 0 duplicate `image_name` entries were detected. No rows require removal for duplication.
2. **Value Bounds Verification:** 0 invalid age values (`<0` or `>120`), 0 invalid gender codes (outside `[0, 1]`), and 0 invalid race codes (outside `[0, 1, 2, 3, 4]`) were detected. All numeric attributes strictly conform to expected boundaries.
3. **Disk File Existence:** 0 broken or missing file paths were found. All 23,705 image filepaths point to valid, readable image files on disk.

*Note on Identifier Columns:* The `image_name` and `filepath` columns are retained as metadata identifiers required for downstream image processing and facial landmark extraction. They are not ML model features and will be separated during feature selection prior to modeling.

In [5]:
# Store initial row count
initial_row_count = len(df)
removal_reasons = {}

# Conditionally apply row removal ONLY if issues were actually detected
df_cleaned = df.copy()

if name_duplicates_count > 0:
    before_cnt = len(df_cleaned)
    df_cleaned = df_cleaned.drop_duplicates(subset=['image_name'])
    removal_reasons["Duplicate image_name entries"] = before_cnt - len(df_cleaned)

if len(invalid_age_df) > 0:
    before_cnt = len(df_cleaned)
    df_cleaned = df_cleaned[(df_cleaned['age'] >= 0) & (df_cleaned['age'] <= 120)]
    removal_reasons["Invalid age out of bounds"] = before_cnt - len(df_cleaned)

if len(invalid_gender_df) > 0:
    before_cnt = len(df_cleaned)
    df_cleaned = df_cleaned[df_cleaned['gender'].isin([0, 1])]
    removal_reasons["Invalid gender code"] = before_cnt - len(df_cleaned)

if len(invalid_race_df) > 0:
    before_cnt = len(df_cleaned)
    df_cleaned = df_cleaned[df_cleaned['race'].isin([0, 1, 2, 3, 4])]
    removal_reasons["Invalid race code"] = before_cnt - len(df_cleaned)

if len(missing_filepaths_df) > 0:
    before_cnt = len(df_cleaned)
    df_cleaned = df_cleaned[df_cleaned['filepath'].apply(os.path.exists)]
    removal_reasons["Missing image file on disk"] = before_cnt - len(df_cleaned)

# Calculate final counts
final_row_count = len(df_cleaned)
total_records_removed = initial_row_count - final_row_count

# Print Before / After summary
print(f"After cleaning shape: {df_cleaned.shape}")
print("\n--- BEFORE / AFTER CLEANING SUMMARY ---")
print(f"Before cleaning: {initial_row_count} rows")
print(f"After cleaning: {final_row_count} rows")
print(f"Records removed: {total_records_removed}")
print("Reasons:")
if removal_reasons:
    for reason_desc, count_removed in removal_reasons.items():
        print(f"  - {reason_desc}: {count_removed} rows")
else:
    print("  - None (0 issues identified across all checks; dataset is fully clean)")

After cleaning shape: (23705, 5)

--- BEFORE / AFTER CLEANING SUMMARY ---
Before cleaning: 23705 rows
After cleaning: 23705 rows
Records removed: 0
Reasons:
  - None (0 issues identified across all checks; dataset is fully clean)


In [6]:
# Export cleaned DataFrame to data/processed/utkface_cleaned.csv
cleaned_csv_path = "data/processed/utkface_cleaned.csv"
df_cleaned.to_csv(cleaned_csv_path, index=False)

print(f"Cleaned dataset saved successfully to '{cleaned_csv_path}' ({len(df_cleaned)} rows).")

Cleaned dataset saved successfully to 'data/processed/utkface_cleaned.csv' (23705 rows).
